# 08. Analyses de fond

Ce notebook regroupe des **analyses de robustesse et de sensibilité** :
- Sensibilité au déséquilibre du corpus (sous-échantillonnage stratifié)
- Comparaison systématique Panel B4 vs corpus complet
- Limites et interprétation

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import CORPUS_V3, RESULTS_DIR, FIGURES_DIR, BLOC_ORDER, BLOC_COLORS, add_events, format_dates

RES_DIR = Path("../data/results")
RES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save(name):
    p = FIGURES_DIR / f"{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 1. Déséquilibre du corpus par bloc

Le corpus est déséquilibré : la Gauche radicale contribue à une part majoritaire des textes. Ce biais affecte les distances cosinus, les fighting words et les conclusions sur la « stabilité » des blocs minoritaires.

In [ ]:
df = pd.read_parquet(CORPUS_V3)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)
df_valid = df[df["bloc"].isin(BLOC_ORDER)].copy()

n_by_bloc = df_valid.groupby("bloc").size().reindex(BLOC_ORDER)
pct = (n_by_bloc / n_by_bloc.sum() * 100).round(1)
tab = pd.DataFrame({"n": n_by_bloc, "pct": pct})
print(tab.to_string())
n_min = n_by_bloc.min()
print(f"\nBloc le plus sous-représenté : n = {int(n_min)}")

## 2. Sensibilité : sous-échantillonnage stratifié

On sous-échantillonne chaque bloc à n_min textes pour tester la robustesse des résultats (distance cosinus, stance moyen) face au déséquilibre.

In [ ]:
n_min = int(n_by_bloc.min())
np.random.seed(42)
df_balanced = (
    df_valid.groupby("bloc", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), n_min), random_state=42))
)

stance_col = "stance_v3" if "stance_v3" in df_balanced.columns else "stance"
if stance_col in df_balanced.columns:
    mean_balanced = df_balanced.groupby("bloc")[stance_col].mean().reindex(BLOC_ORDER)
    mean_full = df_valid.groupby("bloc")[stance_col].mean().reindex(BLOC_ORDER)
    diff = (mean_balanced - mean_full).round(3)
    comp = pd.DataFrame({"complet": mean_full, "sous_echantillonne": mean_balanced, "ecart": diff})
    print("Stance moyen par bloc : corpus complet vs sous-échantillonné (n_min par bloc)")
    print(comp.to_string())
    print("\n→ Si les écarts sont faibles, les conclusions sont robustes au déséquilibre.")

In [ ]:
# Figure : stance moyen corpus complet vs sous-échantillonné
if stance_col in df_balanced.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(BLOC_ORDER))
    w = 0.35
    ax.bar(x - w/2, mean_full.values, w, label="Complet", color=[BLOC_COLORS[b] for b in BLOC_ORDER], alpha=0.8)
    ax.bar(x + w/2, mean_balanced.values, w, label="Sous-échantillonné (n_min)", color=[BLOC_COLORS[b] for b in BLOC_ORDER], alpha=0.4, hatch="//")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([b.replace(" ", "\n") for b in BLOC_ORDER])
    ax.set_ylabel("Stance moyen")
    ax.legend()
    ax.set_title("Robustesse au déséquilibre : stance par bloc")
    save("fig48_sensibilite_corpus")

## 3. Panel B4 vs corpus complet

Le Panel B4 restreint aux députés les plus actifs. Comparaison des distributions de stance.

In [ ]:
# Stance ribbon : Panel B4 vs corpus complet (côte à côte)
panel_path = RES_DIR / "panel_b4.csv"
if panel_path.exists():
    panel = pd.read_csv(panel_path)
    panel_authors = set(panel["author"].tolist())
    df_b4 = df_valid[df_valid["author"].isin(panel_authors)].copy()
    
    stance_full = df_valid.groupby(["month", "bloc"])["stance_v3"].mean().reset_index()
    stance_b4 = df_b4.groupby(["month", "bloc"])["stance_v3"].mean().reset_index()
    
    for name, sm in [("Complet", stance_full), ("Panel B4", stance_b4)]:
        sm["month_ts"] = pd.to_datetime(sm["month"] + "-01")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, sm, tit in [(ax1, stance_full, "Corpus complet"), (ax2, stance_b4, "Panel B4 (44 députés)")]:
        for bloc in BLOC_ORDER:
            sub = sm[sm["bloc"] == bloc]
            if len(sub) > 0:
                ax.plot(sub["month_ts"], sub["stance_v3"], label=bloc, color=BLOC_COLORS[bloc], lw=2)
        add_events(ax)
        format_dates(ax)
        ax.axhline(0, color="gray", lw=0.8)
        ax.set_title(tit)
        ax.legend(loc="upper right", fontsize=8)
        ax.set_ylabel("Stance moyen")
    fig.suptitle("Comparaison stance mensuel : complet vs panel B4")
    plt.tight_layout()
    save("fig49_panel_b4_vs_complet_ribbon")

In [ ]:
panel_path = RES_DIR / "panel_b4.csv"
stance_panel_path = RES_DIR / "stance_panel_vs_complet.csv"

if panel_path.exists():
    panel = pd.read_csv(panel_path)
    n_panel = len(panel)
    print(f"Panel B4 : {n_panel} députés")
    print(panel["bloc"].value_counts().to_string())

if stance_panel_path.exists():
    sp = pd.read_csv(stance_panel_path)
    print("\nComparaison stance : panel vs complet")
    print(sp.head(10).to_string())

## 4. Robustesse 4 blocs vs 5 blocs

Les données agrègent LR et RN en un seul bloc « Droite ». Une analyse 5 blocs (LR et RN séparés) nécessiterait une colonne `groupe` ou `parti` plus granulaire dans le corpus. Les résultats actuels (4 blocs) restent interprétables : la Droite unie présente une position plus homogène que le Centre, ce qui renforce la lecture « extrêmes stables vs centre réactif ».

## 5. Synthèse des limites

| Limite | Impact |
|--------|--------|
| **Déséquilibre corpus** | La Gauche radicale domine en volume → sous-échantillonnage testé ; conclusions robustes si écarts faibles |
| **Panel B4** | Biais de sélection vers les députés les plus actifs ; conclusions sur trajectoires individuelles à interpréter comme telles |
| **Annotation LLM** | Pas de validation humaine ; scores cohérents (accord v3-v4 élevé) mais non validés par des annotateurs |
| **Event studies** | Shift temporel descriptif ; pas de design causal (groupe contrôle, tendances parallèles) — associations observées, pas de causalité |
| **4 blocs** | LR et RN regroupés ; analyse 5 blocs non possible avec les données actuelles |

**Pistes futures** : validation humaine (échantillon stratifié), corpus équilibré, granularité parti pour LR/RN.